In [1]:
from federated_rsf.models import LocalRandomSurvivalForest, FederatedRandomSurvivalForest
from federated_rsf.schema import DatasetSchema, SchemaAligner, SchemaCreator
from federated_rsf.testing import federate_data
from sksurv.datasets import load_veterans_lung_cancer, load_breast_cancer,load_gbsg2
from sksurv.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split


dataset_collectors = [load_breast_cancer, load_gbsg2]
dataset_names = ["_".join(collector.__name__.split("_")[1:]) for collector in dataset_collectors]
n_clients = len(dataset_collectors)

X_list, Y_list, schema_list = [], [], []
for collector, name in zip(dataset_collectors, dataset_names):
    X, Y = collector()
    X = OneHotEncoder().fit_transform(X)
    column_map = dict()
    if "breast_cancer" in name:
        X["size"] = 10* X["size"]
        column_map["size"] = "tsize"

    X_list.append(X)
    Y_list.append(Y)
    schema_list.append(DatasetSchema(X.columns, column_map))

schema_creator = SchemaCreator()
federated_schemas = schema_creator.fit_transform(schema_list)

X_aligned_list = []
local_schema_aligners = []
for X_local, schema in zip(X_list, federated_schemas):
    aligner = SchemaAligner().fit(schema)
    X_aligned = aligner.transform(X_local)
    X_aligned_list.append(X_aligned)
    local_schema_aligners.append(aligner)


In [2]:
X_aligned_list[0].head()

,X200726_at,X200965_s_at,X201068_s_at,X201091_s_at,X201288_at,X201368_at,X201663_s_at,X201664_at,X202239_at,X202240_at,...,grade=intermediate,grade=poorly differentiated,grade=unkown,horTh=yes,menostat=Post,pnodes,progrec,tgrade=II,tgrade=III,tsize
0,10.926361,8.962608,11.630078,10.964107,11.518305,12.038527,9.623518,9.814798,10.016732,7.847383,...,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,30.0
1,12.242090,9.531718,12.626106,11.594716,12.317659,10.776911,10.604577,10.704329,10.161838,8.744875,...,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,30.0
2,11.661716,10.238680,12.572919,9.166088,11.698658,11.353333,9.384927,10.161654,10.032721,8.125487,...,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,25.0
3,12.174021,9.819279,12.109888,9.086937,13.132617,11.859394,8.400839,8.670721,10.727427,8.650810,...,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,18.0
4,11.484011,11.489233,11.779285,8.887616,10.429663,11.401139,7.741092,8.642018,9.556686,8.478862,...,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,30.0


In [3]:
X_aligned_list[1].head()

,X200726_at,X200965_s_at,X201068_s_at,X201091_s_at,X201288_at,X201368_at,X201663_s_at,X201664_at,X202239_at,X202240_at,...,grade=intermediate,grade=poorly differentiated,grade=unkown,horTh=yes,menostat=Post,pnodes,progrec,tgrade=II,tgrade=III,tsize
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,1.0,3.0,48.0,1.0,0.0,21.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,1.0,7.0,61.0,1.0,0.0,12.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,1.0,9.0,52.0,1.0,0.0,35.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,1.0,4.0,60.0,1.0,0.0,17.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,1.0,1.0,26.0,1.0,0.0,35.0


In [21]:
local_models = [LocalRandomSurvivalForest(random_state=0,n_estimators=300 , max_depth=1, update_method='constant') for _ in range(n_clients)]

X_trains, X_tests, Y_trains, Y_tests = [], [], [], []

for X_local, Y_local, local_model in zip(X_aligned_list, Y_list, local_models):

    X_train, X_test, Y_train, Y_test = train_test_split(X_local, Y_local, test_size=0.3, random_state=0)
    X_trains.append(X_train)
    X_tests.append(X_test)
    Y_trains.append(Y_train)
    Y_tests.append(Y_test)

    local_model.fit(X_train, Y_train)



In [22]:
federated_model = FederatedRandomSurvivalForest(local_models=local_models)
federated_model.distribute_trees()

[LocalRandomSurvivalForest(update_method='constant'),
 LocalRandomSurvivalForest(update_method='constant')]

In [23]:

for i, (local_model, X_test, Y_test) in enumerate(zip(local_models, X_tests, Y_tests)):
    local_model.use_local_estimators()
    c_index = local_model.score(X_test, Y_test)
    print(f"Client {i+1}")
    print(f"Local model C-index: {c_index:.4f}")
    local_model.use_federated_estimators()
    federated_c_index = local_model.score(X_test, Y_test)
    print(f"Federated model C-index: {federated_c_index:.4f}")
    print(f"Number of estimators in federated model: {local_model.n_estimators}")
    print("-" * 30)

Client 1
Local model C-index: 0.6628
Federated model C-index: 0.6234
Number of estimators in federated model: 300
------------------------------
Client 2
Local model C-index: 0.6645
Federated model C-index: 0.6647
Number of estimators in federated model: 300
------------------------------
